# Gaussian Mixture Models (GMM) Classification

**Gaussian Mixture Models (GMM)** are probabilistic generative models capable of approximating complex, multimodal probability distributions through a convex combination of $G$ multivariate Gaussian components:

$$f(x) = \sum_{g=1}^{G} w_g \, \mathcal{N}(x \mid \mu_g, \Sigma_g), \quad \text{with } \sum_{g=1}^{G} w_g = 1, \; w_g \geq 0$$

In this notebook we use GMMs to model the class-conditional probability density $P(x \mid C)$ for each class of the Iris dataset, and perform multiclass classification via the Bayes Decision Rule (maximum a posteriori).

In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy
import sklearn
from scripts.utils import *
from scripts.GMM import *

In [5]:
D, L, _, _ = init_dataset()
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)

Le classi sono : ['setosa' 'versicolor' 'virginica'] e le features :['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


To train GMMs in a stable manner and avoid covariance singularity issues (when a Gaussian collapses onto a single data point), we implemented a modular pipeline in [`scripts/GMM.py`](../scripts/GMM.py):

1. **Joint evaluation (`logpdf_GMM`)**: Computes the joint log-likelihoods $\log (w_g \mathcal{N}(x \mid \mu_g, \Sigma_g))$ and the marginal log-density using `scipy.special.logsumexp` to prevent numerical underflow.
2. **Constrained Expectation-Maximization (`EM_constrained`)**:
   - **E-Step**: Computes the responsibilities $\gamma_{ig} = P(g \mid x_i)$.
   - **M-Step**: Updates the sufficient statistics ($Z_g, F_g, S_g$), weights $w_g$, means $\mu_g$, and covariance matrices $\Sigma_g$.
   - **Eigenvalue Flooring ($\psi$)**: Constrains the eigenvalues of $\Sigma_g$ via SVD ($s_i = \max(s_i, \psi)$) to ensure the matrix remains positive definite and bounded below.
3. **Linde-Buzo-Gray Splitting (`LBG_constrained`)**: Hierarchical initialization starting from $G=1$ (global Gaussian) and splitting components along the leading eigenvector direction ($d = \alpha \sqrt{\lambda_1} u_1$) until reaching the target number of components $G \in \{1, 2, 4, 8, 16\}$.

Below we train an independent GMM for each class on the training data and evaluate the **Error Rate** on the validation set as a function of the number of components.

In [6]:
n_components = [1, 2, 4, 8, 16]

# 1. Separazione del dataset di training per classe
DTR0 = DTR[:, LTR == 0]
DTR1 = DTR[:, LTR == 1]
DTR2 = DTR[:, LTR == 2]

for i in n_components:
    print(f"GMM Components: {i}")
    
    # 2. Addestramento di un GMM SEPARATO per ogni singola classe
    gmm0 = LBG_constrained(DTR0, threshold=1e-6, target_components=i, alpha=0.1, psi=0.01)
    gmm1 = LBG_constrained(DTR1, threshold=1e-6, target_components=i, alpha=0.1, psi=0.01)
    gmm2 = LBG_constrained(DTR2, threshold=1e-6, target_components=i, alpha=0.1, psi=0.01)
    # 3. Calcolo delle log-densità marginali sul validation set (ignoriamo la matrice S)
    _, logdens0 = logpdf_GMM(DVAL, gmm0)
    _, logdens1 = logpdf_GMM(DVAL, gmm1)
    _, logdens2 = logpdf_GMM(DVAL, gmm2)

    # 4. Calcolo delle Log-Likelihood Ratios (LLR per classificazione binaria)
    
    SPost=np.vstack([logdens0,logdens1,logdens2])
    
    # 5. Predizione: se LLR > 0 vince la classe 1, altrimenti classe 0
    predictions = np.argmax(SPost,axis=0)
    
    # Calcolo dell'accuratezza e dell'Error Rate
    acc = np.where(predictions == LVAL, 1, 0).mean()
    err = 1 - acc
    
    # Stampa formattata per maggiore chiarezza
    print(f"Error rate: {err * 100:.2f}%\n")

GMM Components: 1
Error rate: 4.00%

GMM Components: 2
Error rate: 4.00%

GMM Components: 4
Error rate: 4.00%

GMM Components: 8
Error rate: 4.00%

GMM Components: 16
Error rate: 4.00%



- **Performance with $G=1$**: The 1-component model corresponds to a standard Gaussian classifier (Tied/Untied); on the validation set it already achieves a competitive error rate of **4.00%** (only 2 misclassified samples out of 50).
- **Stability as components increase**: Thanks to eigenvalue flooring ($\psi = 0.01$) and LBG regularization, the EM algorithm does not suffer from covariance matrix degeneration even when increasing components up to $G = 16$.